Enable GPU and install packages

In [ ]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


Mount Drive and check GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import torch

BASE_PATH = Path("/content/drive/MyDrive/AI-Resume-Screener")

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


Load data

In [ ]:
import pandas as pd

train_df = pd.read_parquet(BASE_PATH / "data/processed/train.parquet")
val_df = pd.read_parquet(BASE_PATH / "data/processed/validation.parquet")

TRAIN_SIZE = min(20000, len(train_df))
VAL_SIZE = min(3000, len(val_df))

train_small = train_df.sample(TRAIN_SIZE, random_state=42).reset_index(drop=True)
val_small = val_df.sample(VAL_SIZE, random_state=42).reset_index(drop=True)

print(train_small.shape, val_small.shape)

(20000, 7) (3000, 7)


Tokenize raw text

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_ds = Dataset.from_pandas(
    train_small[["resume", "jd", "score"]],
    preserve_index=False,
)
val_ds = Dataset.from_pandas(
    val_small[["resume", "jd", "score"]],
    preserve_index=False,
)

def tokenize_batch(batch):
    return tokenizer(
        batch["resume"],
        batch["jd"],
        truncation=True,
        max_length=512,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.rename_column("score", "labels")
val_ds = val_ds.rename_column("score", "labels")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Create regression model and metrics

In [ ]:
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.clip(predictions.reshape(-1), 0, 1)
    labels = labels.reshape(-1)

    correlation = (
        pearsonr(labels, predictions).statistic
        if len(np.unique(labels)) > 1
        else 0.0
    )

    return {
        "mae": mean_absolute_error(labels, predictions),
        "rmse": mean_squared_error(labels, predictions) ** 0.5,
        "pearson": correlation,
    }

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train BERT

In [ ]:
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(BASE_PATH / "models/bert/checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="pearson",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Mae,Rmse,Pearson
1,0.099052,0.034411,0.100118,0.184699,0.795813
2,0.053542,0.025621,0.080426,0.159331,0.849649
3,0.041289,0.023987,0.073249,0.153906,0.861626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=3750, training_loss=0.07176907196044922, metrics={'train_runtime': 1693.2399, 'train_samples_per_second': 35.435, 'train_steps_per_second': 2.215, 'total_flos': 1.578652157952e+16, 'train_loss': 0.07176907196044922, 'epoch': 3.0})

Save model and training log

In [ ]:
FINAL_MODEL_PATH = BASE_PATH / "models/bert/final"

trainer.save_model(str(FINAL_MODEL_PATH))
tokenizer.save_pretrained(str(FINAL_MODEL_PATH))

pd.DataFrame(trainer.state.log_history).to_csv(
    BASE_PATH / "outputs/bert_training_log.csv",
    index=False,
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]